# Python Enumerations — A Second Advanced Tutorial: 20 NEW Problems with Worked Solutions

**Subject:** customizing and extending `enum.Enum` — instance methods, special methods, ordering, equality/hash invariants, truthiness, behavior-only inheritance, multi-field `__new__`, value lookup, and `http.HTTPStatus`.

**Format:** small observation → reasoning → incremental implementation → checks → explanation. This is a **different problem set** from the previous 22 exercises: new situations, new implementations, and new tests.

**Requirements:** Python 3.10+; standard library only; run cells **top to bottom**. Each problem names its classes independently so the notebook can be rerun from a clean kernel. The notebook avoids modifying a member after creation or mixing unrelated enum types accidentally.

**How to study:** first read the *problem* and *prediction* paragraphs; predict the output; run the following cell; only then read the explanation and continue to the solution. All `assert` statements are executable checks, not pseudocode.

**Source vs additions:** The accompanying lesson supplies the central mechanics (`Enum` as a class; dunders; `total_ordering`; truthiness; member-free base classes; `__new__` and `_value_`; `HTTPStatus`). Additional exercises on `_missing_`, `_generate_next_value_`, immutable state, and API-boundary parsing are clearly extensions of those mechanics.

In [1]:
from enum import Enum, unique, auto
from functools import total_ordering
from dataclasses import dataclass
from datetime import timedelta
from http import HTTPStatus
from decimal import Decimal

print('Standard-library imports ready.')

Standard-library imports ready.


# Part I — Members are objects with behavior

The input lesson starts from a deceptively powerful idea: an enum is still a class and its members are instances. We start with methods and representation, introducing one modification per cell.

## Problem 01 — A delivery method that lives on each member

**The problem.** Build `DeliverySpeed` with EXPRESS, STANDARD, and ECONOMY. Each member must calculate a delivery date offset with optional processing days. Reject booleans, negative days, and nonintegers.

**Design target.** A method bound to each singleton; input validation stays inside the method rather than in callers.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Observe the members

Ordinary member values represent transit days. They are not new enum instances on every access.

In [2]:
class DeliverySpeed(Enum):
    EXPRESS = 1
    STANDARD = 3
    ECONOMY = 7

assert DeliverySpeed.EXPRESS is DeliverySpeed.EXPRESS
assert DeliverySpeed(3) is DeliverySpeed.STANDARD
print([(member.name, member.value) for member in DeliverySpeed])

[('EXPRESS', 1), ('STANDARD', 3), ('ECONOMY', 7)]


**Pause and predict.** What object will receive `self` when `DeliverySpeed.STANDARD.estimate(2)` is called? Run the next cell to check your reasoning.

### Step 2 — Add a single focused method

In the solution, guard the type carefully: `isinstance(True, int)` is true, but `True` is not an appropriate number of processing days.

In [3]:
class DeliverySpeed(Enum):
    EXPRESS = 1
    STANDARD = 3
    ECONOMY = 7

    def estimate(self, processing_days: int = 0) -> timedelta:
        if type(processing_days) is not int or processing_days < 0:
            raise ValueError('processing_days must be a nonnegative integer')
        return timedelta(days=self.value + processing_days)

assert DeliverySpeed.STANDARD.estimate(2) == timedelta(days=5)
assert DeliverySpeed.ECONOMY.estimate() == timedelta(days=7)

### Step 3 — Test the boundary rather than only the happy path

A correct public method is defined by what it rejects as well as what it returns.

In [4]:
for bad in (-1, 2.5, True, '3', None):
    try:
        DeliverySpeed.EXPRESS.estimate(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f'accepted invalid processing days: {bad!r}')
print('Five invalid inputs rejected.')

Five invalid inputs rejected.


**What we learned.** The method receives the member as `self`, and uses its stable `.value`. Treat enum members as shared objects, not places to store per-shipment mutable data.

## Problem 02 — Human text versus debugging representation

**The problem.** Design `ConnectionMode` with a lowercase string for user-facing output and a diagnostic representation displaying the class, member, and value.

**Design target.** Keep `__str__` and `__repr__` separate; change display without changing `.value`, equality, or lookup.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — See the untouched defaults

First, check which display function runs when you call `str` and `repr`.

In [5]:
class ConnectionMode(Enum):
    ONLINE = 'connected'
    OFFLINE = 'disconnected'

print('str:', str(ConnectionMode.ONLINE))
print('repr:', repr(ConnectionMode.ONLINE))
assert ConnectionMode.ONLINE.value == 'connected'

str: ConnectionMode.ONLINE
repr: <ConnectionMode.ONLINE: 'connected'>


### Step 2 — Override the two methods independently

The solution deliberately uses the enum name in `repr` for diagnostics and the wire value in `str` for humans.

In [6]:
class ConnectionMode(Enum):
    ONLINE = 'connected'
    OFFLINE = 'disconnected'

    def __str__(self) -> str:
        return self.value

    def __repr__(self) -> str:
        return f'{type(self).__name__}.{self.name}<{self.value!r}>'

assert str(ConnectionMode.OFFLINE) == 'disconnected'
assert repr(ConnectionMode.ONLINE) == "ConnectionMode.ONLINE<'connected'>"

### Step 3 — Check the invariant that should not move

A display change should not alter enum conversion or identity.

In [7]:
assert ConnectionMode('connected') is ConnectionMode.ONLINE
assert ConnectionMode['OFFLINE'] is ConnectionMode.OFFLINE
assert ConnectionMode.ONLINE != 'connected'
print(ConnectionMode.ONLINE, repr(ConnectionMode.ONLINE))

connected ConnectionMode.ONLINE<'connected'>


**What we learned.** Keep persistent identifiers in `.value` and treat `str`/`repr` as presentation. `repr` is diagnostic text here, not a promise of an evaluable Python expression.

## Problem 03 — Add a tiny formatting language

**The problem.** Users of a measurement API want either abbreviated units (`m`) or long names (`meter`), and should get a clear failure for unknown format specifiers.

**Design target.** Implement a deliberately small `__format__` protocol, then show how an f-string dispatches to it.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Establish the usual f-string display

Without a custom formatter, choosing a custom format such as `:short` is not an enum feature we can assume exists.

In [8]:
class LengthUnit(Enum):
    METERS = ('m', 'meter')
    FEET = ('ft', 'foot')

assert LengthUnit.METERS.value == ('m', 'meter')
print(f'Base object: {LengthUnit.METERS}')

Base object: LengthUnit.METERS


### Step 2 — Solve with explicit accepted format specs

An empty spec uses the short form. The string `long` selects the descriptive form. Everything else fails early.

In [9]:
class LengthUnit(Enum):
    METERS = ('m', 'meter')
    FEET = ('ft', 'foot')

    def __format__(self, spec: str) -> str:
        if spec in ('', 'short'):
            return self.value[0]
        if spec == 'long':
            return self.value[1]
        raise ValueError(f'unsupported unit format: {spec!r}')

assert f'{LengthUnit.METERS}' == 'm'
assert f'{LengthUnit.FEET:long}' == 'foot'

### Step 3 — Verify errors and explain the tradeoff

The custom formatter does **not** implement Python alignment/width. That is intentional; supporting width would require parsing and forwarding an additional mini-language.

In [10]:
try:
    f'{LengthUnit.METERS:wide}'
except ValueError as exc:
    assert 'unsupported unit format' in str(exc)
else:
    raise AssertionError('unknown format accepted')

print(f'Report: {LengthUnit.METERS:long} / {LengthUnit.FEET:short}')

Report: meter / ft


**What we learned.** Implementing a dunder is an API decision. Specify supported behavior and reject invalid inputs rather than silently producing misleading output.

## Problem 04 — Keep runtime state separate from members

**The problem.** Demonstrate why one enum member should not hold a different counter for every user session. Move mutable state into a separate dataclass while retaining enum methods.

**Design target.** Preserve member identity and model state as a distinct instance. This problem extends the lesson’s “members are instances” principle.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Inspect shared identity

Two references to a member refer to the same object. Mutable fields added to that object would therefore be shared.

In [11]:
class SessionKind(Enum):
    INTERACTIVE = 'interactive'
    BATCH = 'batch'

first_reference = SessionKind.INTERACTIVE
second_reference = SessionKind.INTERACTIVE
assert first_reference is second_reference

### Step 2 — Model a per-session counter correctly

The enum describes a *kind*; a frozen dataclass describes one particular session. An operation returns a new state object.

In [12]:
@dataclass(frozen=True)
class Session:
    kind: SessionKind
    actions: int = 0

    def next_action(self) -> 'Session':
        return Session(self.kind, self.actions + 1)

session_a = Session(SessionKind.INTERACTIVE)
session_b = Session(SessionKind.INTERACTIVE)
session_a2 = session_a.next_action()
assert session_a2.actions == 1
assert session_a.actions == 0
assert session_b.actions == 0

### Step 3 — Prove the same enum member supports independent states

No per-session data needs to be written to a singleton member.

In [13]:
assert session_a.kind is session_b.kind is SessionKind.INTERACTIVE
assert session_a2 != session_b
print('A:', session_a2, '| B:', session_b)

A: Session(kind=<SessionKind.INTERACTIVE: 'interactive'>, actions=1) | B: Session(kind=<SessionKind.INTERACTIVE: 'interactive'>, actions=0)


**What we learned.** A member models a fixed set of alternatives. Independent changing facts belong in ordinary application objects, not attributes attached to a shared enum singleton.

# Part II — Comparison, equality, hashing, and truthiness

The source lesson introduces rich comparisons, `@total_ordering`, custom equality, and a crucial hashability trap. We build these carefully and test rules that appear only at the boundaries.

## Problem 05 — Diagnose ordering before defining it

**The problem.** A ranking enum contains LOW, MEDIUM, HIGH. First show that ordinary `Enum` cannot order the members, then define a type-safe `<`.

**Design target.** Do not make numeric values accidentally imply comparability to raw integers.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Demonstrate the default failure

`Enum` has identity-based equality, but no standard `<` ordering between its members.

In [14]:
class AlertRank(Enum):
    LOW = 10
    MEDIUM = 30
    HIGH = 90

try:
    AlertRank.LOW < AlertRank.HIGH
except TypeError:
    print('As expected: a plain Enum has no ordering.')
else:
    raise AssertionError('unexpected default ordering')

As expected: a plain Enum has no ordering.


### Step 2 — Provide `__lt__` and reject unrelated operands correctly

Returning `NotImplemented` allows Python to attempt reflected comparison. It is **not** the same as returning `False`.

In [15]:
class AlertRank(Enum):
    LOW = 10
    MEDIUM = 30
    HIGH = 90

    def __lt__(self, other):
        if type(self) is type(other):
            return self.value < other.value
        return NotImplemented

assert AlertRank.LOW < AlertRank.HIGH
assert AlertRank.HIGH > AlertRank.MEDIUM

### Step 3 — Test foreign operands

A rank is not comparable to the integer whose numeric representation happens to match it.

In [16]:
for other in (90, 'HIGH', None):
    try:
        _ = AlertRank.LOW < other
    except TypeError:
        pass
    else:
        raise AssertionError(f'comparison accepted {other!r}')
print('Foreign comparisons raise TypeError.')

Foreign comparisons raise TypeError.


**What we learned.** Ordering is a domain rule, not an automatic side effect of storing numbers. `NotImplemented` is the cooperative rich-comparison protocol.

## Problem 06 — Finish rich comparisons using one primitive

**The problem.** The preceding class has `<` and reflected `>`, but users also need `<=` and `>=`. Avoid writing four nearly identical methods.

**Design target.** Use `@total_ordering` and inherited enum equality. Keep operand validation inside `<`.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Show the gap in the previous implementation

Having a `<` method does not automatically generate all the other operators.

In [17]:
try:
    _ = AlertRank.LOW <= AlertRank.HIGH
except TypeError:
    print('<= is still undefined for the previous class.')
else:
    raise AssertionError('<= unexpectedly supported')

<= is still undefined for the previous class.


### Step 2 — Write the full solution in a small class

`total_ordering` synthesizes the missing rich comparison operators from `__eq__` and `__lt__`.

In [18]:
@total_ordering
class AccessLevel(Enum):
    GUEST = 1
    MEMBER = 2
    ADMIN = 3

    def __lt__(self, other):
        if type(self) is type(other):
            return self.value < other.value
        return NotImplemented

assert AccessLevel.GUEST <= AccessLevel.MEMBER
assert AccessLevel.ADMIN >= AccessLevel.MEMBER
assert AccessLevel.ADMIN > AccessLevel.GUEST
assert AccessLevel.MEMBER != AccessLevel.GUEST

### Step 3 — Probe reflexivity and rejection

Check all four outcomes for an equal member, as well as an unsupported integer operand.

In [19]:
member = AccessLevel.MEMBER
assert member <= member
assert member >= member
assert not (member < member)
assert not (member > member)
try:
    _ = member >= 2
except TypeError:
    pass
else:
    raise AssertionError('ordering against int unexpectedly succeeded')

**What we learned.** `@total_ordering` reduces duplication. It does not fix a faulty comparison policy, so write one correct primitive and test every generated relation.

## Problem 07 — Order by domain rank, not lexical value

**The problem.** A review pipeline uses wire values `draft`, `approved`, `in_review`. Alphabetical ordering is not workflow ordering.

**Design target.** Keep stable wire values but compare using an explicit rank table.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — See the misleading ordering

Lexicographic comparison of the values says nothing about workflow sequence.

In [20]:
wire_values = ['draft', 'in_review', 'approved']
assert sorted(wire_values) != wire_values
print('Alphabetical:', sorted(wire_values))

Alphabetical: ['approved', 'draft', 'in_review']


### Step 2 — Implement the domain rule

Compute rank with a mapping keyed by the wire *value*. The table is fixed here, and comparisons remain limited to this enum class.

In [21]:
@total_ordering
class ReviewStep(Enum):
    DRAFT = 'draft'
    IN_REVIEW = 'in_review'
    APPROVED = 'approved'

    @property
    def rank(self) -> int:
        return {'draft': 0, 'in_review': 1, 'approved': 2}[self.value]

    def __lt__(self, other):
        if type(self) is type(other):
            return self.rank < other.rank
        return NotImplemented

assert ReviewStep.DRAFT < ReviewStep.IN_REVIEW < ReviewStep.APPROVED

### Step 3 — Show that sorting obeys the model

Sorting compares enum members using the custom rich comparison; serialization still takes the original wire value.

In [22]:
unordered = [ReviewStep.APPROVED, ReviewStep.DRAFT, ReviewStep.IN_REVIEW]
assert sorted(unordered) == [ReviewStep.DRAFT, ReviewStep.IN_REVIEW, ReviewStep.APPROVED]
assert ReviewStep.IN_REVIEW.value == 'in_review'
print([step.value for step in sorted(unordered)])

['draft', 'in_review', 'approved']


**What we learned.** The natural comparison key belongs to the *business meaning* of the member, not necessarily to its stored value or declaration order.

## Problem 08 — Understand why overriding equality can make an enum unhashable

**The problem.** Build a deliberately broken equality method in a temporary enum, inspect its hashability, then design a safe API without changing equality.

**Design target.** A dictionary key must have a hash compatible with equality. Do not “repair” the issue by inventing inconsistent hashing.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Construct a small counterexample

Python intentionally disables default hashing when a class overrides `__eq__` but does not define a matching `__hash__`.

In [23]:
class BrokenTag(Enum):
    FIRST = 1
    SECOND = 2

    def __eq__(self, other):
        if type(self) is type(other):
            return self.value == other.value
        return NotImplemented

assert BrokenTag.FIRST == BrokenTag.FIRST
assert BrokenTag.__hash__ is None

### Step 2 — Observe the concrete failure

The member is no longer suitable as a set element or dictionary key.

In [24]:
try:
    {BrokenTag.FIRST: 'item'}
except TypeError as exc:
    assert 'unhashable' in str(exc)
    print('Expected:', exc)
else:
    raise AssertionError('broken class was unexpectedly hashable')

Expected: unhashable type: 'BrokenTag'


### Step 3 — Solve with default equality and a separate comparison function

When the goal is checking an integer code, there is no need to replace enum equality at all.

In [25]:
class SafeTag(Enum):
    FIRST = 1
    SECOND = 2

    def has_code(self, code: int) -> bool:
        return type(code) is int and self.value == code

labels = {SafeTag.FIRST: 'first entry', SafeTag.SECOND: 'second entry'}
assert labels[SafeTag.FIRST] == 'first entry'
assert SafeTag.FIRST.has_code(1)
assert not SafeTag.FIRST.has_code(True)
assert SafeTag.FIRST != 1

**What we learned.** The default enum equality and hash contract is already consistent. Prefer a named `has_code` operation for comparisons across a type boundary.

## Problem 09 — Protect against the int/bool equality collision

**The problem.** You receive numeric inputs from a caller and discover that in Python `True == 1`. Demonstrate why blindly passing values into an integer-valued Enum can be surprising.

**Design target.** Write a strict parsing function that rejects `bool` without modifying enum equality or hash behavior.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Make the collision visible

This equality is a property of Python integers and booleans, not a bug specific to `Enum`.

In [26]:
assert True == 1
class PacketKind(Enum):
    DATA = 1
    CONTROL = 2

assert PacketKind(True) is PacketKind.DATA
print('Naive lookup accepted True as numeric code 1.')

Naive lookup accepted True as numeric code 1.


### Step 2 — Define a strict boundary

Use `type(...) is int` if the input contract is *exactly* a built-in integer, not a bool or arbitrary integer subclass.

In [27]:
def parse_packet_kind(raw: object) -> PacketKind:
    if type(raw) is not int:
        raise TypeError('packet kind must be a built-in integer')
    try:
        return PacketKind(raw)
    except ValueError as exc:
        raise ValueError(f'unknown packet kind: {raw}') from exc

assert parse_packet_kind(1) is PacketKind.DATA
assert parse_packet_kind(2) is PacketKind.CONTROL

### Step 3 — Separate invalid types from unknown codes

Type mistakes and unknown-but-well-formed numbers should have different errors.

In [28]:
for bad in (True, 1.0, '1', None):
    try:
        parse_packet_kind(bad)
    except TypeError:
        pass
    else:
        raise AssertionError(f'accepted wrong type {bad!r}')
try:
    parse_packet_kind(99)
except ValueError:
    pass
else:
    raise AssertionError('accepted unknown code')

**What we learned.** Conversion is a boundary operation. Validate the external representation before calling the enum constructor; default `Enum` semantics remain intact.

## Problem 10 — Give truthiness one explicit meaning

**The problem.** An enum has DISABLED = 0 and ENABLED = 1. Inspect ordinary truthiness, then implement `__bool__` for a domain where zero should be false.

**Design target.** Understand the default first; never assume `bool(member)` is `bool(member.value)` without an override.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Observe the default surprise

A plain Enum member is true regardless of whether `.value` is zero or an empty string.

In [29]:
class FeatureDefault(Enum):
    DISABLED = 0
    ENABLED = 1

assert bool(FeatureDefault.DISABLED) is True
assert bool(FeatureDefault.ENABLED) is True

### Step 2 — Define semantic truthiness

An intentional `__bool__` conveys the rule for *this exact* binary enum.

In [30]:
class FeatureToggle(Enum):
    DISABLED = 0
    ENABLED = 1

    def __bool__(self) -> bool:
        return self is FeatureToggle.ENABLED

assert bool(FeatureToggle.DISABLED) is False
assert bool(FeatureToggle.ENABLED) is True

### Step 3 — Use it and discuss readability

The boolean expression works; explicit identity may still make code clearer in security- or safety-sensitive conditions.

In [31]:
def feature_label(toggle: FeatureToggle) -> str:
    if not isinstance(toggle, FeatureToggle):
        raise TypeError('expected FeatureToggle')
    return 'on' if toggle else 'off'

assert feature_label(FeatureToggle.ENABLED) == 'on'
assert feature_label(FeatureToggle.DISABLED) == 'off'

**What we learned.** Default truthiness is independent of member values. Override it only when the entire enum has a clear yes/no interpretation.

## Problem 11 — Prefer an explicit predicate for a multistate enum

**The problem.** A device has IDLE, WARMING, RUNNING, and FAILED. There is no universal truthiness interpretation for all four states.

**Design target.** Leave `__bool__` unchanged and add a `can_serve` property that precisely states the action being checked.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Define all four states

The stored numeric code is not equivalent to permission to serve traffic.

In [32]:
class DevicePhase(Enum):
    IDLE = 0
    WARMING = 1
    RUNNING = 2
    FAILED = 3

assert all(bool(phase) for phase in DevicePhase)

### Step 2 — Put behavior on the enum where it belongs

A named property communicates the intent of the test without altering the semantics of `if phase`.

In [33]:
class DevicePhase(Enum):
    IDLE = 0
    WARMING = 1
    RUNNING = 2
    FAILED = 3

    @property
    def can_serve(self) -> bool:
        return self is DevicePhase.RUNNING

assert DevicePhase.RUNNING.can_serve
assert not DevicePhase.WARMING.can_serve
assert not DevicePhase.IDLE.can_serve
assert bool(DevicePhase.IDLE) is True

### Step 3 — Compose the predicate into a function

Validate the caller-provided object instead of relying on its truthiness.

In [34]:
def route_request(phase: DevicePhase) -> str:
    if not isinstance(phase, DevicePhase):
        raise TypeError('phase must be DevicePhase')
    return 'route' if phase.can_serve else 'hold'

assert route_request(DevicePhase.RUNNING) == 'route'
assert route_request(DevicePhase.FAILED) == 'hold'

**What we learned.** A domain-specific predicate is often clearer than a global truthiness override. Pick semantics at the level of the question your code is answering.

# Part III — Extending enumeration behavior without extending members

An enum with members cannot be subclassed to add new enum members. A member-free base class can, however, hold reusable behavior. Two exercises turn that rule into a design technique.

## Problem 12 — Prove why extending a populated enum fails

**The problem.** A team wants to add `UNKNOWN` to an established direction enum by subclassing it. Diagnose the failure, then replace the design with a member-free behavior base.

**Design target.** No introspection hacks, mutation of existing enum members, or exception swallowing.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Try the tempting but unsupported design

This class creation is expected to raise `TypeError`.

In [35]:
class CardinalDirection(Enum):
    NORTH = 'north'
    SOUTH = 'south'

try:
    class ExpandedDirection(CardinalDirection):
        UNKNOWN = 'unknown'
except TypeError:
    print('Cannot extend an enum that already declares members.')
else:
    raise AssertionError('extended a populated enum')

Cannot extend an enum that already declares members.


### Step 2 — Extract only behavior into a base

A subclass of a member-free `Enum` may introduce its own concrete members.

In [36]:
class DescribedEnum(Enum):
    def describe(self) -> str:
        return f'{type(self).__name__}.{self.name} uses {self.value!r}'

class CompassPoint(DescribedEnum):
    N = 'north'
    S = 'south'
    E = 'east'
    W = 'west'

assert 'north' in CompassPoint.N.describe()
assert CompassPoint('west') is CompassPoint.W

### Step 3 — Build a second enum from the same behavior

Reusing instance methods does not mix the two sets of members.

In [37]:
class WindDirection(DescribedEnum):
    TAILWIND = 'tail'
    HEADWIND = 'head'

assert 'WindDirection' in WindDirection.TAILWIND.describe()
assert CompassPoint.N != WindDirection.TAILWIND
print(WindDirection.HEADWIND.describe())

WindDirection.HEADWIND uses 'head'


**What we learned.** Subclass an *empty* enumeration to inherit behavior, then give each concrete subclass its own members. You cannot append enum members by normal subclassing.

## Problem 13 — Create reusable ordering without cross-catalog comparisons

**The problem.** Two different catalogues each have numeric ranking values. A shared ordering base must not compare one catalogue’s members with the other’s.

**Design target.** Define strict type-checked ordering once and reuse it through two subclasses.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Define the behavior-only base

Use `type(self) is type(other)` to keep the ordering relation local to the concrete catalogue.

In [38]:
@total_ordering
class SameTypeOrdered(Enum):
    def __lt__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self.value < other.value

### Step 2 — Instantiate two independent orderings

One set is numeric; the other uses ordered tuples. The shared method relies only on comparable member values *within* the same enum.

In [39]:
class AuditLevel(SameTypeOrdered):
    INFO = 1
    WARNING = 2
    ERROR = 3

class GridDimension(SameTypeOrdered):
    POINT = (0,)
    LINE = (0, 1)
    PLANE = (0, 1, 2)

assert AuditLevel.WARNING < AuditLevel.ERROR
assert GridDimension.POINT <= GridDimension.PLANE

### Step 3 — Test domain isolation and hashability

`total_ordering` does not replace enum equality. Default identity equality and hashing remain available.

In [40]:
try:
    _ = AuditLevel.INFO < GridDimension.POINT
except TypeError:
    pass
else:
    raise AssertionError('cross-catalogue ordering succeeded')

assert {AuditLevel.INFO: 'message'}[AuditLevel.INFO] == 'message'
assert AuditLevel.INFO != GridDimension.POINT

**What we learned.** An abstract behavior-only enum is a powerful reuse mechanism; ensure it rejects unrelated concrete enums instead of comparing coincidentally equal values.

# Part IV — Multi-field values and richer lookup

The lesson’s `HTTPStatus`-inspired exercise is about separating the primary value from metadata. We begin by finding the tuple-value failure, then build and verify the correct scalar-valued design. The final lookup exercises are explicitly additional applications.

## Problem 14 — Find the tuple-value lookup mistake

**The problem.** A measurement code has a numeric identifier, display label, and precision. A simple tuple-valued enum exposes all three but cannot be looked up using only the number.

**Design target.** Diagnose before fixing: understand the difference between a tuple-valued member and a scalar-valued member with metadata.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Use the most direct declaration

No `__new__` means the entire tuple becomes `.value`.

In [41]:
class TupleMetric(Enum):
    DISTANCE = (101, 'Distance', 2)
    TIME = (102, 'Time', 3)

assert TupleMetric.DISTANCE.value == (101, 'Distance', 2)
assert TupleMetric((101, 'Distance', 2)) is TupleMetric.DISTANCE

### Step 2 — Investigate the unexpected lookup failure

The constructor matches the *whole value*, not one field inside the value.

In [42]:
try:
    TupleMetric(101)
except ValueError:
    print('Numeric code 101 is not equal to the entire tuple.')
else:
    raise AssertionError('numeric lookup unexpectedly succeeded')

Numeric code 101 is not equal to the entire tuple.


### Step 3 — Add convenient properties, but observe what they cannot fix

Properties expose tuple positions but do not change the primary underlying value.

In [43]:
class TupleMetric(Enum):
    DISTANCE = (101, 'Distance', 2)
    TIME = (102, 'Time', 3)

    @property
    def code(self) -> int:
        return self.value[0]

    @property
    def label(self) -> str:
        return self.value[1]

assert TupleMetric.DISTANCE.code == 101
assert TupleMetric.TIME.label == 'Time'
assert isinstance(TupleMetric.DISTANCE.value, tuple)

**What we learned.** Properties offer nicer access to metadata but cannot change value lookup. For `Metric(101)`, the member’s actual `.value` must be `101`.

## Problem 15 — Construct a scalar-valued, multi-field enum

**The problem.** Correct the metric declaration using `__new__`: integer code becomes `.value`, while label and decimal places remain attributes.

**Design target.** Implement deterministic construction and strict validation at enum definition time.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Identify construction responsibilities

Each tuple on the right-hand side is expanded into `__new__(cls, code, label, places)`. Use `object.__new__` for a plain `Enum` and set `_value_` explicitly.

In [44]:
class MetricCode(Enum):
    DISTANCE = (101, 'Distance', 2)
    TIME = (102, 'Time', 3)
    MASS = (103, 'Mass', 1)

    def __new__(cls, code: int, label: str, places: int):
        if type(code) is not int or code < 0:
            raise ValueError('code must be a nonnegative integer')
        if not isinstance(label, str) or not label.strip():
            raise ValueError('label must be nonblank text')
        if type(places) is not int or not 0 <= places <= 6:
            raise ValueError('places must be an integer in 0..6')
        member = object.__new__(cls)
        member._value_ = code
        member.label = label
        member.places = places
        return member

### Step 2 — Verify lookup and metadata independently

The code is now scalar, but the metadata has not disappeared.

In [45]:
assert MetricCode(101) is MetricCode.DISTANCE
assert MetricCode.MASS.value == 103
assert MetricCode.MASS.label == 'Mass'
assert MetricCode.TIME.places == 3
assert MetricCode['DISTANCE'] is MetricCode.DISTANCE

### Step 3 — Put a useful instance method on top

The precision is part of the catalogue, so a member method can apply it to a numeric value. `Decimal` helps us express decimal rounding in a predictable way.

In [46]:
def format_metric(metric: MetricCode, raw: Decimal) -> str:
    if not isinstance(metric, MetricCode):
        raise TypeError('expected MetricCode')
    if not isinstance(raw, Decimal) or not raw.is_finite():
        raise TypeError('raw must be a finite Decimal')
    quantum = Decimal(1).scaleb(-metric.places)
    return f'{raw.quantize(quantum):.{metric.places}f} {metric.label}'

assert format_metric(MetricCode.DISTANCE, Decimal('12.345')) == '12.34 Distance'
assert format_metric(MetricCode.MASS, Decimal('4.44')) == '4.4 Mass'

### Step 4 — Check construction-time validation using an intentionally invalid class

Creating an invalid catalogue should fail before the application can use it.

In [47]:
try:
    class BadMetric(Enum):
        BAD = (9, 'Bad', 99)
        def __new__(cls, code, label, places):
            if not 0 <= places <= 6:
                raise ValueError('invalid places')
            obj = object.__new__(cls)
            obj._value_ = code
            return obj
except ValueError:
    pass
else:
    raise AssertionError('invalid metric was declared')

**What we learned.** `_value_` governs lookup; regular attributes retain extra fields. Validate fields during construction so a broken enum declaration never imports successfully.

## Problem 16 — Avoid silent aliases when metadata disagrees

**The problem.** Two command descriptions accidentally use the same primary code with different labels. A normal enum may silently alias them.

**Design target.** Combine custom `_value_` assignment with `@unique` so the catalogue rejects duplicate primary codes.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Observe alias behavior in a small example

Two names with the same value ordinarily denote one member; the alias does not keep independent per-name metadata.

In [48]:
class AliasSample(Enum):
    FIRST = 8
    SECOND = 8

assert AliasSample.FIRST is AliasSample.SECOND
assert len(list(AliasSample)) == 1
assert len(AliasSample.__members__) == 2

### Step 2 — Construct the problematic catalogue and inspect the failure

The duplicate arises from assigning the *numeric code* to `_value_`. Use the `@unique` decorator to detect it.

In [49]:
try:
    @unique
    class DuplicateCommand(Enum):
        START = (8, 'begin')
        RESUME = (8, 'continue')

        def __new__(cls, code, label):
            member = object.__new__(cls)
            member._value_ = code
            member.label = label
            return member
except ValueError:
    print('Duplicate primary code rejected at definition time.')
else:
    raise AssertionError('duplicate code accepted')

Duplicate primary code rejected at definition time.


### Step 3 — Write a valid version and test every member

Uniqueness is a schema check, not a replacement for runtime validation of inbound codes.

In [50]:
@unique
class CommandCode(Enum):
    START = (8, 'begin')
    RESUME = (9, 'continue')

    def __new__(cls, code, label):
        member = object.__new__(cls)
        member._value_ = code
        member.label = label
        return member

assert len(CommandCode) == 2
assert CommandCode(9).label == 'continue'

**What we learned.** The equality of primary `.value` fields determines aliases. Validate uniqueness *after* assigning `_value_`, particularly when extra metadata differs.

## Problem 17 — Normalize friendly inputs without changing stored values

**The problem.** A connector receives location names with surrounding spaces and mixed letter case. Add friendly lookup while preserving lowercase stable wire values.

**Design target.** Extend `Enum` with `_missing_` and avoid accepting non-string values or partial matches. **Extension beyond the source lesson.**

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — See what ordinary value lookup requires

Ordinary lookup matches the actual value exactly.

In [51]:
class BasicRegion(Enum):
    NORTH = 'north'
    SOUTH = 'south'

try:
    BasicRegion(' NORTH ')
except ValueError:
    print('Ordinary lookup is exact.')
else:
    raise AssertionError('normalization happened unexpectedly')

Ordinary lookup is exact.


### Step 2 — Implement controlled normalization

Only strings are eligible for case folding and trimming. Iterate over members and return the matching singleton.

In [52]:
class RegionName(Enum):
    NORTH = 'north'
    SOUTH = 'south'
    CENTRAL = 'central'

    @classmethod
    def _missing_(cls, value):
        if not isinstance(value, str):
            return None
        normalized = value.strip().casefold()
        for member in cls:
            if member.value == normalized:
                return member
        return None

assert RegionName(' NORTH ') is RegionName.NORTH
assert RegionName('CeNtRaL') is RegionName.CENTRAL

### Step 3 — Reject ambiguous or invalid data

Normalization does not mean fuzzy matching, arbitrary coercion, or accepting an empty string.

In [53]:
for bad in (2, None, '', 'north-west', 'cent'):
    try:
        RegionName(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f'accepted {bad!r}')
assert RegionName.NORTH.value == 'north'

**What we learned.** `_missing_` is an optional customization hook for failed lookups. The canonical member values remain unchanged, and the acceptance policy stays explicit.

## Problem 18 — Generate codes predictably with auto-numbering

**The problem.** A catalogue needs stable codes at intervals of ten: 10, 20, 30. Replace repeated numeric assignments with a class-level generator, and test the result.

**Design target.** Override `_generate_next_value_`, distinguish construction behavior from instance methods, and warn against accidental renumbering. **Extension beyond the source lesson (which mentions auto numbering as a later topic).**

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Inspect ordinary `auto` behavior

The usual generated integers start at one for a basic Enum; this might not match a protocol’s numbering scheme.

In [54]:
class DefaultEventCode(Enum):
    OPEN = auto()
    CLOSE = auto()

assert [member.value for member in DefaultEventCode] == [1, 2]

### Step 2 — Implement the generator before the member declarations

The hook gets the member name, start, count, and previously generated values. Using `count` makes the assignment independent of each member’s name.

In [55]:
class TenStepCode(Enum):
    def _generate_next_value_(name, start, count, last_values):
        return (count + 1) * 10

    OPEN = auto()
    PAUSE = auto()
    CLOSE = auto()

assert [member.value for member in TenStepCode] == [10, 20, 30]
assert TenStepCode(20) is TenStepCode.PAUSE

### Step 3 — Make protocol stability a test

Auto-numbering depends on declaration positions: adding a member in the middle may change all later codes. Test externally published values explicitly.

In [56]:
EXPECTED_CODES = {'OPEN': 10, 'PAUSE': 20, 'CLOSE': 30}
actual = {member.name: member.value for member in TenStepCode}
assert actual == EXPECTED_CODES
print('Published codes match the expected contract:', actual)

Published codes match the expected contract: {'OPEN': 10, 'PAUSE': 20, 'CLOSE': 30}


**What we learned.** A generator saves repetition but is safe for external protocols only when published numeric mappings are tested and carefully versioned.

# Part V — A standard-library example and a guided capstone

`http.HTTPStatus` is a practical example from the lesson: numeric values and human-readable phrases coexist. We apply that pattern, then create a separate catalogue with real, validated data boundaries.

## Problem 19 — Analyze HTTPStatus without treating every integer as a valid status

**The problem.** Build a tiny response formatter that accepts an existing `HTTPStatus` member or a strictly validated integer code; show its phrase and decide its broad HTTP class.

**Design target.** Distinguish input validation, enum lookup, and presentation. Do not subclass `HTTPStatus`, which already has members.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — Inspect the relevant properties

Members are enum instances with numeric `.value` and a readable `.phrase`.

In [57]:
not_found = HTTPStatus.NOT_FOUND
assert not_found.value == 404
assert not_found.name == 'NOT_FOUND'
assert not_found.phrase == 'Not Found'
assert HTTPStatus(404) is not_found
print(not_found.value, not_found.phrase)

404 Not Found


### Step 2 — Define narrow conversion from external input

An existing member is accepted directly. Otherwise, the input must be exactly an integer, and the code must exist in the standard enum.

In [58]:
def coerce_http_status(raw: object) -> HTTPStatus:
    if isinstance(raw, HTTPStatus):
        return raw
    if type(raw) is not int:
        raise TypeError('HTTP status must be HTTPStatus or an integer')
    return HTTPStatus(raw)

assert coerce_http_status(200) is HTTPStatus.OK
assert coerce_http_status(HTTPStatus.NOT_FOUND) is HTTPStatus.NOT_FOUND

### Step 3 — Add presentation while keeping codes stable

The hundreds digit identifies the broad HTTP class for recognized statuses. The formatter works on validated enum members only.

In [59]:
HTTP_CLASSES = {
    1: 'informational', 2: 'success', 3: 'redirection',
    4: 'client error', 5: 'server error',
}

def explain_http_status(raw: object) -> dict:
    member = coerce_http_status(raw)
    return {
        'code': member.value,
        'phrase': member.phrase,
        'category': HTTP_CLASSES[member.value // 100],
    }

assert explain_http_status(404) == {
    'code': 404, 'phrase': 'Not Found', 'category': 'client error'
}
assert explain_http_status(HTTPStatus.CREATED)['category'] == 'success'

### Step 4 — Exercise both error categories

The standard catalogue may not recognize arbitrary three-digit numbers, and bool should not be accepted as an integer.

In [60]:
for wrong_type in (True, 404.0, '404', None):
    try:
        explain_http_status(wrong_type)
    except TypeError:
        pass
    else:
        raise AssertionError(f'accepted wrong input {wrong_type!r}')
try:
    explain_http_status(999)
except ValueError:
    pass
else:
    raise AssertionError('accepted unknown HTTP status')

**What we learned.** `HTTPStatus` demonstrates a useful enum design: the lookup value is the numeric code while text is additional member information. Check external types before value lookup.

## Problem 20 — Capstone: compile a typed telemetry event catalogue

**The problem.** A telemetry packet arrives as a dictionary containing an integer `code` and a numeric `reading`. Each event kind has an identifier, text label, display precision, and urgency. Build a member-free base, a scalar-valued enum, safe parsing, canonical output, sorting, and robust tests.

**Design target.** Integrate methods, `__new__`, `_value_`, total ordering, strict type checks, immutable per-packet state, and zero accidental cross-type comparison. This is a fresh telemetry design, not the prior job-lifecycle capstone.

Work down the small cells in order. Each numbered step establishes one fact before the next step changes anything.

### Step 1 — State the invariants before coding

The enum member is the **kind of event**, not the event instance. Codes are unique positive integers. Labels cannot be blank. Precision must lie in 0..4. Urgency is a nonnegative integer. Input `True` is not a valid numeric code or reading.

**Input contract.** The exact input mapping must have keys `code` and `reading`. `code` is a built-in `int` and must resolve to a known event kind. `reading` is a finite, built-in `int` or `float` (not `bool`). The decoded result is an immutable packet. The output contains a numeric code, a canonical numeric reading, and a display message.

**Prediction:** Do two packets with the same kind have to share their readings? No. The kind is an enum singleton; readings belong to separate immutable packet objects.

### Step 2 — Write one reusable behavior-only enum

The base supplies type-safe ordering by urgency and a human-readable display line. It defines **no members**, so subclassing it is supported.

In [61]:
@total_ordering
class TelemetryBase(Enum):
    def __lt__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self.urgency < other.urgency

    def display(self, reading: float) -> str:
        return f'{self.label}: {reading:.{self.places}f}'

### Step 3 — Implement the member constructor and catalogue

The declaration’s right-hand tuple provides four inputs. The primary integer becomes `_value_`; label, precision and urgency are attached once during enum construction.

In [62]:
@unique
class TelemetryKind(TelemetryBase):
    TEMPERATURE = (201, 'Temperature', 1, 1)
    PRESSURE = (202, 'Pressure', 2, 3)
    VIBRATION = (203, 'Vibration', 3, 2)

    def __new__(cls, code, label, places, urgency):
        if type(code) is not int or code <= 0:
            raise ValueError('code must be a positive integer')
        if not isinstance(label, str) or not label.strip():
            raise ValueError('label must not be blank')
        if type(places) is not int or not 0 <= places <= 4:
            raise ValueError('invalid display precision')
        if type(urgency) is not int or urgency < 0:
            raise ValueError('invalid urgency')
        obj = object.__new__(cls)
        obj._value_ = code
        obj.label = label
        obj.places = places
        obj.urgency = urgency
        return obj

assert TelemetryKind(202) is TelemetryKind.PRESSURE
assert TelemetryKind.PRESSURE.display(3.14159) == 'Pressure: 3.14'

### Step 4 — Confirm comparison semantics and a design constraint

Urgencies here are unique. If you allow equal urgencies, different members can satisfy neither `<` nor `>`, while remaining unequal; `@total_ordering` then yields `<=` as false. That is a *partial* ranking, not a total order. A robust total order needs a tie-breaker.

In [63]:
# Use a deterministic two-field key for the final total-order design.
def telemetry_sort_key(kind: TelemetryKind) -> tuple[int, int]:
    if not isinstance(kind, TelemetryKind):
        raise TypeError('expected TelemetryKind')
    return kind.urgency, kind.value

assert sorted(TelemetryKind, key=telemetry_sort_key) == [
    TelemetryKind.TEMPERATURE, TelemetryKind.VIBRATION, TelemetryKind.PRESSURE
]
assert TelemetryKind.TEMPERATURE < TelemetryKind.PRESSURE
assert {TelemetryKind.TEMPERATURE: 'temp'}[TelemetryKind.TEMPERATURE] == 'temp'

### Step 5 — Write the immutable packet record

Two packets can have the same kind and still have independent readings. We normalize to `float` after checking type and finiteness.

In [64]:
import math

@dataclass(frozen=True)
class TelemetryPacket:
    kind: TelemetryKind
    reading: float

    def __post_init__(self):
        if not isinstance(self.kind, TelemetryKind):
            raise TypeError('kind must be TelemetryKind')
        if type(self.reading) not in (int, float):
            raise TypeError('reading must be a built-in int or float')
        try:
            numeric = float(self.reading)
        except OverflowError as exc:
            raise ValueError('reading is outside finite float range') from exc
        if not math.isfinite(numeric):
            raise ValueError('reading must be finite')
        object.__setattr__(self, 'reading', numeric)

p1 = TelemetryPacket(TelemetryKind.TEMPERATURE, 20)
p2 = TelemetryPacket(TelemetryKind.TEMPERATURE, 21)
assert p1.kind is p2.kind and p1.reading != p2.reading
assert p1.reading == 20.0

### Step 6 — Validate and decode at the API boundary

Do not pass arbitrary payloads directly into the enum. Validate the mapping shape and exact code type, then perform scalar member lookup.

In [65]:
def decode_telemetry(raw: object) -> TelemetryPacket:
    if type(raw) is not dict or set(raw) != {'code', 'reading'}:
        raise TypeError('expected dict with exactly code and reading')
    if type(raw['code']) is not int:
        raise TypeError('code must be a built-in int')
    try:
        kind = TelemetryKind(raw['code'])
    except ValueError as exc:
        raise ValueError(f'unknown telemetry code: {raw["code"]}') from exc
    return TelemetryPacket(kind, raw['reading'])

packet = decode_telemetry({'code': 202, 'reading': 101.237})
assert packet.kind is TelemetryKind.PRESSURE
assert packet.reading == 101.237

### Step 7 — Generate a canonical output representation

Serialization should use `.value` and an explicit message, not `repr(member)` or a tuple of internal attributes.

In [66]:
def encode_telemetry(packet: TelemetryPacket) -> dict:
    if not isinstance(packet, TelemetryPacket):
        raise TypeError('expected TelemetryPacket')
    return {
        'code': packet.kind.value,
        'reading': packet.reading,
        'message': packet.kind.display(packet.reading),
    }

summary = encode_telemetry(packet)
assert summary == {
    'code': 202, 'reading': 101.237, 'message': 'Pressure: 101.24'
}
assert decode_telemetry({'code': summary['code'], 'reading': summary['reading']}) == packet

### Step 8 — Build an adversarial test table

The negative cases are part of the solution. Reject invalid shapes, wrong types, unknown codes, bools, and non-finite readings.

In [67]:
bad_shapes = ([], None, {}, {'code': 201},
              {'code': 201, 'reading': 1, 'extra': 'x'})
for raw in bad_shapes:
    try:
        decode_telemetry(raw)
    except TypeError:
        pass
    else:
        raise AssertionError(f'accepted invalid shape: {raw!r}')

for code_value in (True, 201.0, '201'):
    try:
        decode_telemetry({'code': code_value, 'reading': 1})
    except TypeError:
        pass
    else:
        raise AssertionError(f'accepted invalid code type: {code_value!r}')

for invalid_reading in (True, '1.2', None):
    try:
        decode_telemetry({'code': 201, 'reading': invalid_reading})
    except TypeError:
        pass
    else:
        raise AssertionError(f'accepted invalid reading: {invalid_reading!r}')

for nonfinite in (float('nan'), float('inf'), -float('inf')):
    try:
        decode_telemetry({'code': 201, 'reading': nonfinite})
    except ValueError:
        pass
    else:
        raise AssertionError('accepted a non-finite reading')

try:
    decode_telemetry({'code': 999, 'reading': 1})
except ValueError:
    pass
else:
    raise AssertionError('accepted unknown code')
print('Negative tests passed.')

Negative tests passed.


### Step 9 — Run a miniature end-to-end example

The dictionary simulates an input message; sorting the enum catalogue is independent of processing incoming packets.

In [68]:
incoming = [
    {'code': 201, 'reading': 20.55},
    {'code': 203, 'reading': 0.123456},
    {'code': 202, 'reading': 101.237},
]
packets = [decode_telemetry(item) for item in incoming]
reports = [encode_telemetry(item) for item in packets]
assert [item['code'] for item in reports] == [201, 203, 202]
for report in reports:
    print(report['message'])

Temperature: 20.6
Vibration: 0.123
Pressure: 101.24


**What we learned.** The enum is the immutable catalogue of event kinds. A dataclass holds per-event observations; the boundary performs validation; serialization publishes only stable scalar codes and intentional display text.

# Final synthesis — questions to answer without running code

1. Why is `DeliverySpeed.STANDARD` one object regardless of how often you access it?
2. Why can `MetricCode(101)` succeed but `TupleMetric(101)` fail?
3. Why does defining `__eq__` without a compatible `__hash__` break dictionary keys?
4. When does `NotImplemented` differ from `False` in `__lt__`?
5. Why is `True` a potentially dangerous input for an integer-valued enum?
6. Why can `DescribedEnum` be subclassed but `CardinalDirection` cannot?
7. Why do we store a `TelemetryPacket.reading` on the dataclass rather than on the enum member?
8. What is the risk of adding an `auto()` member in the middle of a published protocol?

**Answer key:** (1) enum members are singletons; (2) `_value_` is a scalar in the corrected implementation but an entire tuple in the naive one; (3) Python disables hashing when custom equality has no matching hash, and hashes must agree for equal objects; (4) `NotImplemented` delegates comparison to the reflected operation and otherwise permits a `TypeError`, while `False` is a final boolean answer; (5) booleans compare equal to integers 1 and 0; (6) only an enum without members can be subclassed for new members; (7) each packet has independent changing data while members describe fixed categories; (8) generated numeric values can shift, breaking compatibility with external consumers.

**Best-practice recap:** validate external input before enum lookup; keep equality/hash consistent; do not order across unrelated enum classes; use `@total_ordering` carefully; choose explicit predicates for multistate enums; subclass member-free bases for shared functionality; assign `_value_` to the true primary lookup key; test canonical codes and negative cases.

**Notebook provenance:** The topic and tutorial cadence follow the user-supplied lesson; all problem scenarios and worked code are newly authored for this notebook. No external packages or network requests are required.